# GEC-RAG Project
## Notebook 04 — T5-GEC Baseline

This notebook runs **T5-GEC** as the non-LLM baseline for our benchmark.

**Model:** `vennify/t5-base-grammar-correction` — a T5-base model fine-tuned
specifically for grammatical error correction, available on HuggingFace.

**Role in the paper:**
T5-GEC represents a dedicated, fine-tuned seq2seq GEC approach — it corrects
errors but produces no explanation. It establishes the correction-quality
baseline against which LLM-based systems are compared.

**What this notebook produces:**
- T5-GEC corrections on the dev set
- F0.5 score via ERRANT
- Per error-type performance breakdown
- Saved predictions for Notebook 07 (evaluation)

---
### Cell 1 — Imports & Setup

In [1]:
import os
import json
import torch
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from transformers import T5ForConditionalGeneration, T5Tokenizer
import warnings
warnings.filterwarnings('ignore')

# ⚠️ Update BASE_DIR to match your actual path
BASE_DIR           = 'C:/gecrag'
DATA_DIR           = os.path.join(BASE_DIR, 'data/write_and_improve')
FINAL_VERSIONS_DIR = os.path.join(DATA_DIR, 'user-prompt-final-versions')
RESULTS_DIR        = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device : {device}')
if device == 'cuda':
    print(f'✅ GPU    : {torch.cuda.get_device_name(0)}')
print(f'✅ Torch  : {torch.__version__}')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

✅ Device : cuda
✅ GPU    : NVIDIA GeForce RTX 4070
✅ Torch  : 2.5.1+cu121


---
### Cell 2 — Load T5-GEC Model
> ⏳ First run downloads the model from HuggingFace (~900MB). Cached after first download.

In [2]:
MODEL_NAME = 'vennify/t5-base-grammar-correction'

print(f'Loading model: {MODEL_NAME}')
print('(First run will download ~900MB — cached after that)\n')

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model     = model.to(device)
model.eval()

print(f'✅ T5-GEC model loaded')
print(f'   Model     : {MODEL_NAME}')
print(f'   Device    : {device}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Quick sanity check
test_input = 'grammar: I go to school yesterday .'
inputs     = tokenizer(test_input, return_tensors='pt').to(device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_length=128)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f'\n🔍 Sanity check:')
print(f'   Input : I go to school yesterday .')
print(f'   Output: {result}')

Loading model: vennify/t5-base-grammar-correction
(First run will download ~900MB — cached after that)



tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

✅ T5-GEC model loaded
   Model     : vennify/t5-base-grammar-correction
   Device    : cuda
   Parameters: 222,903,552

🔍 Sanity check:
   Input : I go to school yesterday .
   Output: I went to school yesterday.


---
### Cell 3 — Load Dev Set

In [3]:
def load_text_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

orig_path = os.path.join(FINAL_VERSIONS_DIR, 'en-writeandimprove2024-final-versions-dev-sentences.orig')
corr_path = os.path.join(FINAL_VERSIONS_DIR, 'en-writeandimprove2024-final-versions-dev-sentences.corr')

try:
    orig_sentences = load_text_file(orig_path)
    corr_sentences = load_text_file(corr_path)
    assert len(orig_sentences) == len(corr_sentences)

    print(f'✅ Dev split loaded')
    print(f'   Sentence pairs: {len(orig_sentences)}')
    print(f'\nSample:')
    for i in range(3):
        print(f'  [{i+1}] ORIG: {orig_sentences[i]}')
        print(f'       CORR: {corr_sentences[i]}')

except FileNotFoundError:
    print('❌ dev.orig not found. Available files:')
    for f in sorted(os.listdir(FINAL_VERSIONS_DIR)):
        print(f'   {f}')
    print('\n👉 Update orig_path and corr_path accordingly')

✅ Dev split loaded
   Sentence pairs: 4352

Sample:
  [1] ORIG: There are lots of important things in our life.
       CORR: There are lots of important things in our life.
  [2] ORIG: However, in most cases, they depend on the priorities we have and acceptation we are awaiting from it.
       CORR: However, in most cases, they depend on the priorities we have and the acceptation we are awaiting from it.
  [3] ORIG: If it fulfils our dreams and aims or not?
       CORR: If it fulfils our dreams and aims or not.


---
### Cell 4 — Run T5-GEC Inference
> ⚠️ Set MAX_SENTENCES=200 for a quick test first, then None for the full run.
> ⏳ Full dev set takes several minutes on GPU.

In [5]:
def run_t5_gec(sentences, model, tokenizer, device, batch_size=16):
    """
    Run T5-GEC inference on a list of sentences.
    T5-GEC uses the prefix 'grammar: ' to trigger correction mode.
    """
    predictions = []
    errors      = 0

    for i in tqdm(
        range(0, len(sentences), batch_size),
        desc='T5-GEC inference'
    ):
        batch = sentences[i:i+batch_size]

        # Add task prefix
        inputs_text = [f'grammar: {s}' for s in batch]

        try:
            inputs = tokenizer(
                inputs_text,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=128
            ).to(device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_length=128,
                    num_beams=5,
                    early_stopping=True
                )

            decoded = tokenizer.batch_decode(
                outputs, skip_special_tokens=True
            )
            predictions.extend(decoded)

        except Exception as e:
            errors += len(batch)
            predictions.extend(batch)  # Keep original on error

    print(f'\n✅ Inference complete')
    print(f'   Processed : {len(predictions)} sentences')
    print(f'   Errors    : {errors}')
    return predictions


# ⚠️ Change to None for full run after test succeeds
MAX_SENTENCES = None

test_orig = orig_sentences[:MAX_SENTENCES] if MAX_SENTENCES else orig_sentences
test_corr = corr_sentences[:MAX_SENTENCES] if MAX_SENTENCES else corr_sentences

print(f'Running T5-GEC on {len(test_orig)} sentences...')
t5_predictions = run_t5_gec(test_orig, model, tokenizer, device)

# Show sample predictions
print('\nSample predictions:')
for i in range(5):
    print(f'\n[{i+1}] ORIGINAL  : {test_orig[i]}')
    print(f'     T5-GEC    : {t5_predictions[i]}')
    print(f'     REFERENCE : {test_corr[i]}')

Running T5-GEC on 4352 sentences...


T5-GEC inference:   0%|          | 0/272 [00:00<?, ?it/s]


✅ Inference complete
   Processed : 4352 sentences
   Errors    : 0

Sample predictions:

[1] ORIGINAL  : There are lots of important things in our life.
     T5-GEC    : There are lots of important things in our life.
     REFERENCE : There are lots of important things in our life.

[2] ORIGINAL  : However, in most cases, they depend on the priorities we have and acceptation we are awaiting from it.
     T5-GEC    : However, in most cases, they depend on the priorities we have and the acceptance we are awaiting from it.
     REFERENCE : However, in most cases, they depend on the priorities we have and the acceptation we are awaiting from it.

[3] ORIGINAL  : If it fulfils our dreams and aims or not?
     T5-GEC    : If it fulfils our dreams and aims, or not?
     REFERENCE : If it fulfils our dreams and aims or not.

[4] ORIGINAL  : First of all, let us start with one of the most important things in our life and that is health.
     T5-GEC    : First of all, let us start with one of 

---
### Cell 5 — Compute F0.5 Score via ERRANT

In [6]:
import errant

annotator = errant.load('en')
print('✅ ERRANT annotator loaded')

def compute_f05(orig_list, pred_list, ref_list, annotator):
    """
    Compute F0.5 score using ERRANT.
    F0.5 weights precision twice as much as recall —
    standard metric for GEC evaluation.
    """
    tp = fp = fn = 0

    for orig, pred, ref in tqdm(
        zip(orig_list, pred_list, ref_list),
        total=len(orig_list),
        desc='Computing F0.5'
    ):
        try:
            orig_p = annotator.parse(orig)
            pred_p = annotator.parse(pred)
            ref_p  = annotator.parse(ref)

            pred_edits = set()
            for e in annotator.annotate(orig_p, pred_p):
                if e.type != 'noop':
                    pred_edits.add((e.o_start, e.o_end, e.c_str))

            ref_edits = set()
            for e in annotator.annotate(orig_p, ref_p):
                if e.type != 'noop':
                    ref_edits.add((e.o_start, e.o_end, e.c_str))

            tp += len(pred_edits & ref_edits)
            fp += len(pred_edits - ref_edits)
            fn += len(ref_edits  - pred_edits)

        except Exception:
            continue

    beta      = 0.5
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f05       = ((1 + beta**2) * precision * recall /
                 (beta**2 * precision + recall)) if (precision + recall) > 0 else 0

    return {
        'precision': round(precision * 100, 2),
        'recall'   : round(recall    * 100, 2),
        'f05'      : round(f05       * 100, 2),
        'tp': tp, 'fp': fp, 'fn': fn,
    }


print('Computing F0.5 for T5-GEC...')
t5_scores = compute_f05(test_orig, t5_predictions, test_corr, annotator)

sep = '='*45
print(f'\n{sep}')
print(f'T5-GEC Baseline Results')
print(sep)
print(f'  Precision : {t5_scores["precision"]}%')
print(f'  Recall    : {t5_scores["recall"]}%')
print(f'  F0.5      : {t5_scores["f05"]}%')
print(f'  TP/FP/FN  : {t5_scores["tp"]}/{t5_scores["fp"]}/{t5_scores["fn"]}')
print(sep)

✅ ERRANT annotator loaded
Computing F0.5 for T5-GEC...


Computing F0.5:   0%|          | 0/4352 [00:00<?, ?it/s]


T5-GEC Baseline Results
  Precision : 48.51%
  Recall    : 25.19%
  F0.5      : 40.93%
  TP/FP/FN  : 2062/2189/6125


---
### Cell 6 — Per Error-Type Performance

In [7]:
def compute_per_type_scores(orig_list, pred_list, ref_list, annotator):
    """
    Compute F0.5 broken down by ERRANT error type.
    Shows which error types the model handles well vs poorly.
    """
    type_stats = {}

    for orig, pred, ref in tqdm(
        zip(orig_list, pred_list, ref_list),
        total=len(orig_list),
        desc='Per-type analysis'
    ):
        try:
            orig_p = annotator.parse(orig)
            pred_p = annotator.parse(pred)
            ref_p  = annotator.parse(ref)

            pred_edits = {}
            for e in annotator.annotate(orig_p, pred_p):
                if e.type != 'noop':
                    pred_edits[(e.o_start, e.o_end, e.c_str)] = e.type

            ref_edits = {}
            for e in annotator.annotate(orig_p, ref_p):
                if e.type != 'noop':
                    ref_edits[(e.o_start, e.o_end, e.c_str)] = e.type

            for key, etype in ref_edits.items():
                if etype not in type_stats:
                    type_stats[etype] = {'tp':0,'fp':0,'fn':0}
                if key in pred_edits:
                    type_stats[etype]['tp'] += 1
                else:
                    type_stats[etype]['fn'] += 1

            for key, etype in pred_edits.items():
                if key not in ref_edits:
                    if etype not in type_stats:
                        type_stats[etype] = {'tp':0,'fp':0,'fn':0}
                    type_stats[etype]['fp'] += 1

        except Exception:
            continue

    beta    = 0.5
    results = []
    for etype, s in type_stats.items():
        p   = s['tp']/(s['tp']+s['fp']) if (s['tp']+s['fp']) > 0 else 0
        r   = s['tp']/(s['tp']+s['fn']) if (s['tp']+s['fn']) > 0 else 0
        f05 = ((1+beta**2)*p*r/(beta**2*p+r)) if (p+r) > 0 else 0
        results.append({
            'error_type': etype,
            'tp': s['tp'], 'fp': s['fp'], 'fn': s['fn'],
            'precision' : round(p*100,   2),
            'recall'    : round(r*100,   2),
            'f05'       : round(f05*100, 2),
        })

    return pd.DataFrame(results).sort_values('f05', ascending=False)


print('Computing per error-type scores...')
t5_per_type = compute_per_type_scores(
    test_orig, t5_predictions, test_corr, annotator
)

print('\nT5-GEC Per Error-Type F0.5 (top 15):')
print(t5_per_type.head(15).to_string(index=False))

Computing per error-type scores...


Per-type analysis:   0%|          | 0/4352 [00:00<?, ?it/s]


T5-GEC Per Error-Type F0.5 (top 15):
  error_type  tp  fp  fn  precision  recall   f05
  R:VERB:SVA 120  44  61      73.17   66.30 71.68
 M:VERB:FORM  37  16  29      69.81   56.06 66.55
     R:SPELL 207  83 229      71.38   47.48 64.85
 R:VERB:FORM 133  58 187      69.63   41.56 61.35
  R:NOUN:NUM  95  53 126      64.19   42.99 58.43
      U:PRON  19  11  28      63.33   40.43 56.89
     U:PUNCT  10   6  14      62.50   41.67 56.82
      R:ORTH 273 165 443      62.33   38.13 55.31
M:VERB:TENSE  18  11  43      62.07   29.51 50.85
      R:PRON  22   8  82      73.33   21.15 49.11
      U:CONJ   6   6   8      50.00   42.86 48.39
       M:DET 103  74 264      58.19   28.07 47.91
      M:PRON  36  29  87      55.38   29.27 47.00
       U:DET  77  54 234      58.78   24.76 46.11
     R:MORPH  39  32 102      54.93   27.66 45.88


---
### Cell 7 — Save Results

In [8]:
# Save predictions
predictions_path = os.path.join(RESULTS_DIR, 't5_predictions.json')
with open(predictions_path, 'w', encoding='utf-8') as f:
    json.dump({
        'system'      : 'T5-GEC',
        'model'       : 'vennify/t5-base-grammar-correction',
        'n_sentences' : len(test_orig),
        'scores'      : t5_scores,
        'predictions' : [
            {
                'original'  : orig,
                'prediction': pred,
                'reference' : ref
            }
            for orig, pred, ref in zip(
                test_orig, t5_predictions, test_corr
            )
        ]
    }, f, ensure_ascii=False, indent=2)
print(f'✅ Predictions saved : {predictions_path}')

# Save per-type scores
per_type_path = os.path.join(RESULTS_DIR, 't5_per_type_scores.csv')
t5_per_type.to_csv(per_type_path, index=False)
print(f'✅ Per-type scores   : {per_type_path}')

# Save to shared baseline scores file
scores_path = os.path.join(RESULTS_DIR, 'baseline_scores.json')
try:
    with open(scores_path, 'r') as f:
        all_scores = json.load(f)
except FileNotFoundError:
    all_scores = {}

all_scores['T5-GEC'] = t5_scores
with open(scores_path, 'w') as f:
    json.dump(all_scores, f, indent=2)
print(f'✅ Baseline scores   : {scores_path}')

✅ Predictions saved : C:/gecrag\results\t5_predictions.json
✅ Per-type scores   : C:/gecrag\results\t5_per_type_scores.csv
✅ Baseline scores   : C:/gecrag\results\baseline_scores.json


---
### Cell 8 — Summary

In [9]:
sep = '='*50
print(sep)
print('T5-GEC BASELINE — SUMMARY')
print(sep)

print(f'\n📊 Model:')
print(f'   vennify/t5-base-grammar-correction')
print(f'   Sentences evaluated: {len(test_orig)}')

print(f'\n📊 Overall Performance:')
print(f'   Precision : {t5_scores["precision"]}%')
print(f'   Recall    : {t5_scores["recall"]}%')
print(f'   F0.5      : {t5_scores["f05"]}%')

specific = t5_per_type[~t5_per_type['error_type'].str.contains('OTHER')]
print(f'\n📊 Top 5 Error Types (best F0.5):')
print(specific.head(5)[['error_type','precision','recall','f05']].to_string(index=False))

print(f'\n📊 Bottom 5 Error Types (hardest for T5-GEC):')
print(specific.tail(5)[['error_type','precision','recall','f05']].to_string(index=False))

print(f'\n⚠️  Reminder: If you ran with MAX_SENTENCES=200,')
print(f'   set MAX_SENTENCES=None in Cell 4 and re-run')
print(f'   cells 4-7 for the full evaluation.')

print(f'\n📁 Output Files:')
print(f'   - results/t5_predictions.json')
print(f'   - results/t5_per_type_scores.csv')
print(f'   - results/baseline_scores.json')

print(f'\n➡️  Next: Run Notebook 05 — LLM Baselines (GPT-4o & Llama 3.1)')

T5-GEC BASELINE — SUMMARY

📊 Model:
   vennify/t5-base-grammar-correction
   Sentences evaluated: 4352

📊 Overall Performance:
   Precision : 48.51%
   Recall    : 25.19%
   F0.5      : 40.93%

📊 Top 5 Error Types (best F0.5):
 error_type  precision  recall   f05
 R:VERB:SVA      73.17   66.30 71.68
M:VERB:FORM      69.81   56.06 66.55
    R:SPELL      71.38   47.48 64.85
R:VERB:FORM      69.63   41.56 61.35
 R:NOUN:NUM      64.19   42.99 58.43

📊 Bottom 5 Error Types (hardest for T5-GEC):
 error_type  precision  recall   f05
      U:ADJ      11.11    8.33 10.42
    M:PUNCT       0.00    0.00  0.00
     R:CONJ       0.00    0.00  0.00
R:VERB:INFL       0.00    0.00  0.00
R:NOUN:POSS       0.00    0.00  0.00

⚠️  Reminder: If you ran with MAX_SENTENCES=200,
   set MAX_SENTENCES=None in Cell 4 and re-run
   cells 4-7 for the full evaluation.

📁 Output Files:
   - results/t5_predictions.json
   - results/t5_per_type_scores.csv
   - results/baseline_scores.json

➡️  Next: Run Notebook 05 —